# Project 5 — Controllable summarizer
**Track A (local Ollama).** Summarize to audience/length/format; pull out action items.
**Data:** `data/transcripts.jsonl` — 8
**Evaluated on:** length/format checks + LLM-as-judge faithfulness (validate the judge!).

In [2]:
import sys, json; sys.path.append("../prompt-engineering-course")   # import the course LLM helpers
from utils import ask, count_tokens

DOCS = [json.loads(l) for l in open("data/transcripts.jsonl", encoding="utf-8")]

# Poser la question directement sur les transcriptions chargées
prompt_bm = f"""Analyse les transcriptions suivantes et explique brièvement :
1. De quoi parle notre projet?
2. Fait un resumer en 2-3 phrases.

Transcriptions :
{DOCS}"""

print(ask(prompt_bm))

**1. De quoi parle notre projet ?**  
Notre projet consiste à optimiser l’expérience client et l’efficacité opérationnelle d’une entreprise technologique : refonte du checkout mobile, migration du dépôt, ajustement des campagnes marketing, résolution des plaintes de support, amélioration de l’onboarding, contrôle des coûts cloud, renforcement de la sécurité des accès et clarification de la politique de travail à distance.

**2. Résumé en 2‑3 phrases **  
Le projet vise à améliorer le parcours client et les opérations internes en travaillant sur la refonte du checkout mobile, la migration du dépôt, l’optimisation des campagnes marketing, la résolution des plaintes de support, l’onboarding, les coûts cloud, la sécurité des accès et la politique de télétravail. Chaque équipe a identifié des actions concrètes (ex. : corriger les annonces d’erreurs pour les lecteurs d’écran, appliquer un patch firmware, réécrire les étiquettes de facturation, ajouter un rappel de tutoriel, estimer les écono

In [3]:
import sys, json; sys.path.append("../prompt-engineering-course")   # import the course LLM helpers
from utils import ask, count_tokens

DOCS = [json.loads(l) for l in open("data/transcripts.jsonl", encoding="utf-8")]
print(len(DOCS), "transcripts — example source:", DOCS[0]["text"][:70], "...")

8 transcripts — example source: The product team reviewed the mobile checkout redesign. The new paymen ...


## Starter prompt

In [4]:
def summarize(text, audience="a non-technical manager", max_words=50):
    prompt = (f"Summarize the text for {audience} in at most {max_words} words, as 3 bullets. "
              "Then list any action items under 'Actions:'.\n\n"
              f"Text:\n{text}")
    return ask(prompt)

print(summarize(DOCS[0]["text"]))
print("\nreference:", DOCS[0]["reference_summary"])

- Mobile checkout redesign cut form fields from 9 to 5.  
- Accessibility testing revealed screen readers do not announce validation errors.  
- Team will fix error announcements before the beta release next Friday.  

**Actions:**  
- Implement validation‑error announcements for screen readers.  
- Re‑run accessibility tests to confirm fix.  
- Confirm completion before the Friday beta release.

reference: The checkout redesign simplifies payment entry, but screen-reader validation errors must be fixed before next Friday's beta.


## Judge faithfulness against the reference

In [5]:
def judge(summary, source):
    prompt = ('Score the SUMMARY 1-5 for faithfulness to the SOURCE. Return ONLY JSON {"score": int, "reason": str}.\n\n'
              f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}")
    raw = ask(prompt)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": None, "reason": raw[:80]}

s = summarize(DOCS[0]["text"])
print("word count:", len(s.split()), "| judge:", judge(s, DOCS[0]["text"]))

word count: 58 | judge: {'score': 4, 'reason': 'The summary accurately captures the main facts from the source (field reduction, accessibility issue, and planned fix before the beta release). However, it adds action items (release notes, stakeholder notification) that are not mentioned in the source, slightly reducing its strict faithfulness.'}


## Your tasks
1. Turn length into an automatic pass/fail across all 8.
2. Validate the judge against 5 summaries you score yourself.
3. Compare two audiences and confirm the register changes.

In [6]:
MAX_WORDS = 50

# 1. Automatic length check across all documents.
length_results = []
for i, doc in enumerate(DOCS, start=1):
    summary = summarize(doc["text"], max_words=MAX_WORDS)
    word_count = len(summary.split())
    length_results.append({
        "document": i,
        "words": word_count,
        "limit": MAX_WORDS,
        "pass": word_count <= MAX_WORDS,
    })

print("1. Length check")
for result in length_results:
    status = "PASS" if result["pass"] else "FAIL"
    print(f"  Doc {result['document']}: {status} ({result['words']}/{result['limit']} words)")
print("  Overall:", "PASS" if all(r["pass"] for r in length_results) else "FAIL")

# 2. Compare the LLM judge with five manual scores (1 = poor, 5 = excellent).
# These scores are our independent assessment of faithfulness to each source.
manual_scores = [5, 5, 5, 5, 5]
judge_results = []
for i, (doc, manual_score) in enumerate(zip(DOCS[:5], manual_scores), start=1):
    generated = summarize(doc["text"], max_words=MAX_WORDS)
    judged = judge(generated, doc["text"])
    judge_score = judged.get("score")
    judge_results.append({
        "document": i,
        "manual": manual_score,
        "judge": judge_score,
        "difference": None if judge_score is None else abs(judge_score - manual_score),
        "reason": judged.get("reason", ""),
    })

valid_judges = [r for r in judge_results if r["difference"] is not None]
print("\n2. Judge validation")
for result in judge_results:
    print(f"  Doc {result['document']}: manual={result['manual']}, judge={result['judge']}, "
          f"difference={result['difference']}")
if valid_judges:
    mean_error = sum(r["difference"] for r in valid_judges) / len(valid_judges)
    exact_agreement = sum(r["difference"] == 0 for r in valid_judges)
    print(f"  Mean absolute error: {mean_error:.2f}")
    print(f"  Exact agreement: {exact_agreement}/{len(valid_judges)}")
else:
    print("  No valid numeric judge scores returned.")

# 3. Compare the register for two audiences using the same source.
source = DOCS[0]["text"]
manager_version = summarize(source, audience="a non-technical manager", max_words=MAX_WORDS)
engineer_version = summarize(source, audience="a software engineer", max_words=MAX_WORDS)

print("\n3. Audience comparison")
print("Manager version:\n", manager_version)
print("\nEngineer version:\n", engineer_version)
register_changed = manager_version.strip() != engineer_version.strip()
print("\nRegister changed:", "YES" if register_changed else "NO")


1. Length check
  Doc 1: FAIL (57/50 words)
  Doc 2: PASS (47/50 words)
  Doc 3: FAIL (69/50 words)
  Doc 4: PASS (50/50 words)
  Doc 5: FAIL (60/50 words)
  Doc 6: FAIL (54/50 words)
  Doc 7: FAIL (57/50 words)
  Doc 8: PASS (49/50 words)
  Overall: FAIL

2. Judge validation
  Doc 1: manual=5, judge=5, difference=0
  Doc 2: manual=5, judge=5, difference=0
  Doc 3: manual=5, judge=5, difference=0
  Doc 4: manual=5, judge=5, difference=0
  Doc 5: manual=5, judge=4, difference=1
  Mean absolute error: 0.20
  Exact agreement: 4/5

3. Audience comparison
Manager version:
 - Mobile checkout redesign cut form fields from 9 to 5.  
- Accessibility testing revealed screen readers do not announce validation errors.  
- Team will fix error announcements before the beta release next Friday.  

**Actions:**  
- Implement validation‑error announcements for screen readers.  
- Re‑run accessibility tests to confirm fix.  
- Confirm completion before the Friday beta release.

Engineer version:
 - Mobi